In [6]:
from IPython.display import Image

- GPU、TPU、SoC（System on Chip）
    - GPU: 干得快，人多力量大（AI 训练目前的主流）。
        - 可能会被集成进 SoC 里
    - TPU（Tensor Processing Unit）: 干得专，术业有专攻（Google 的 AI 秘密武器）。
        - 可能会以 NPU 的形式集成进 SoC 里
        - ASIC：ASIC 指的是为了某种特定的用途而专门设计和制造的芯片。在 AI 领域，ASIC 就是为了解决 Transformer 或 CNN 中的 $Y = WX + b$ 而存在的物理实体。
    - SoC: 装得全，麻雀虽小五脏俱全（手机和现代电脑的核心）。
        - 包含了 CPU + GPU + NPU
- 主存与缓存
    - cpu：ddr & l1/l2/l3 cache
    - gpu：hbm & sram

- 内存条
    - 消费级内存条：面向普通 PC/游戏主机的内存（通常是 UDIMM，不带真正的纠错通道）。价格更低、频率/时序选择多。
    - ECC 内存条：带错误校验和纠正能力（Error-Correcting Code）的内存，主要用于服务器/工作站，稳定性更高、价格更高。

### cpu + ram vs. GPU

$$
\text{1 GPU SM} \approx \text{1 CPU Core (但它是超宽的 SIMD 架构)}
$$
$$
\text{1 CUDA Core} \approx \text{1 CPU ALU Lane}
$$

- CPU + RAM：通常是板级连接（通过主板上的 DDR 插槽），距离远，带宽低，延迟高。
- 消费级（如 4090）：也是板级连接，但使用 GDDR6X，带宽远高于 DDR。
    - GPU (如 RTX 4090)：这是一个自包含的计算子系统（Self-contained Subsystem）。
        - GPU Die (AD102 芯片) $\approx$
        - CPU显存颗粒 (GDDR6X) $\approx$ 内存条
        - 显卡 PCB 板 $\approx$ 主板
        - 显卡散热器 $\approx$ CPU 散热器
    - 为什么显存（VRAM）必须焊在 GPU 旁边？CPU 和 GPU 形态差异最根本的原因：对“带宽”的极致渴求改变了物理形态。
- 位宽（Bus Width）
    - CPU：通常是 128-bit 位宽（双通道）。
    - GPU (4090)：拥有 384-bit 的显存位宽。
- sm vs. cpu core
    - 一个 CPU Core 拥有：独立的指令缓存（I-Cache）、解码器、分支预测、乱序执行引擎（Out-of-Order Execution）、ALU、以及 L1/L2 缓存。
    - 一个 SM 拥有：独立的指令缓存、Warp 调度器（Warp Scheduler）、寄存器文件（Register File）、L1 缓存/共享内存（Shared Memory），以及大量的 CUDA Cores（4090,128），Tensor cores。

In [8]:
Image(url='https://www.notebookcheck.net/fileadmin/Notebooks/NVIDIA/ad103_block_diagram.jpeg', width=600)

### 内存（Volatile）与硬盘（Non-Volatile）

- ram
    - DRAM (Dynamic Random Access Memory)，独立于 cpu 芯片，off-chip
        - DDR, LPDDR, GDDR, HBM
    - SRAM（static ram，CPU/GPU 内部的 L1/L2/L3 Cache）：集成在 cpu 芯片内部，on-die
- 硬盘
    - hdd => sdd (flash 材质)

### DDR, LPDDR, GDDR, HBM

In [4]:
Image(url='./imgs/ram.jpeg', width=600)

- DDR (Double Data Rate SDRAM)
    - CPU 内存，如 DDR4, DDR5
    - mac intel 芯，`system_profiler SPMemoryDataType` 查看自己的内存类型
- LPDDR (Low Power DDR)
    - 通常直接焊在主板上或封装在 SoC 里 (PoP)，不可更换。
    - iPhone、MacBook (Apple Silicon)
    - M 系列芯片的内存是 Unified Memory（Unified Memory Architecture, UMA，统一内存），本质上都是 LPDDR 技术。
        - M1 / M2 系列： 使用的是 LPDDR4X (4266 MHz) 或 LPDDR5 (6400 MHz)。
        - M3 / M4 系列： 通常使用 LPDDR5 或 LPDDR5X。
    - UMA
        - 在传统的 PC 架构（比如 Intel CPU + NVIDIA 显卡）中，内存是被物理隔离的，CPU dram，GPU vram，当 CPU处理完一个数据，需要交给 GPU 去渲染或计算时，CPU 必须把这份数据通过 PCIe 总线“拷贝”一份 传送到 GPU 的显存里。
        - 统一内存架构：拆掉围墙，在 M 系列芯片中，CPU、GPU 和神经网络引擎（NPU）都连接到同一个统一内存池。
            - 当 CPU 处理完数据，它不需要把数据“搬”给 GPU。
            - 它只需要把数据在内存里的地址（指针） 告诉 GPU：“嘿，东西就在桌子中间，你自己拿去用。”
            - GPU 直接读取同一个位置的数据，处理完后，CPU 也能立即看到结果。
- GDDR (Graphics DDR)
    - Graphics DDR (如 GDDR6, GDDR6X, GDDR7)
    - 高频率： 也就是“高时钟速度”。它通过极高的频率来换取大带宽。
- HBM（High Bandwidth Memory）
    - 3D 堆叠： 像盖楼一样，把 DRAM 芯片垂直堆叠起来，通过硅通孔 (TSV) 连接。
    - GDDR 可能只有 32-bit/chip，而 HBM 单堆栈就是 1024-bit。它靠“路宽”取胜，而不是靠“车速”（频率）。

In [5]:
Image(url='./imgs/hbm.webp', width=500)